## Vicsek Model

The particle velocity evolves as,
$$
    \boldsymbol{v}_i = v_0 (\cos{\theta_i}, \sin{\theta_i}). \\
    \theta_{i+1} = \langle\theta_i\rangle_{\text{nearby particles}} + \eta(t)\Delta t
$$
The position changes as,
$$
    \boldsymbol{x}_{i+1} = \boldsymbol{x}_i + \boldsymbol{v}_i \Delta t.
$$
Here $i$ labels the time step. The average is the direction of the nearby particles' mean velocity, and $\eta(t)$ is a fresh random angular rate at each step.


In [ ]:
import numpy as np


In [ ]:
# Initialization (angles in radians).
N = 100          # Number of particles.
L = 10.0         # Box side length.
v_0 = 0.1        # Fixed speed.
R = 1.0          # Neighbor radius.
eta = 0.3        # Random angular-rate width; 0 means no noise.
dt = 1.0
n_steps = 200
seed = 42        # Same seed gives the same simulation.


In [ ]:
# Direction update: align with neighbors, then add random noise.
def update_theta(x, theta, R, L, eta, rng, dt=1.0):
    displacement = x[:, None, :] - x[None, :, :]
    displacement -= L * np.round(displacement / L)  # Periodic distances.
    neighbors = np.sum(displacement**2, axis=2) <= R**2  # Includes self.

    sum_cos = neighbors @ np.cos(theta)
    sum_sin = neighbors @ np.sin(theta)
    average = np.arctan2(sum_sin, sum_cos)
    # If directions cancel exactly, keep the particle's previous heading.
    average = np.where(np.hypot(sum_cos, sum_sin) > 1e-12, average, theta)
    noise = rng.uniform(-eta / 2, eta / 2, size=len(theta)) * dt
    return (average + noise) % (2 * np.pi)


In [ ]:
# Velocity update: only direction changes; speed stays fixed.
def update_velocity(theta, v_0):
    return v_0 * np.column_stack((np.cos(theta), np.sin(theta)))


In [ ]:
# Evolve and store every step.
def vicsek_model(N, L, v_0, R, eta, *, dt=1.0, n_steps=200, seed=None):
    """Return x, v, theta, t for particles in a 2D periodic box.

    x and v: (n_steps + 1, N, 2); theta: (n_steps + 1, N).
    eta is the random angular-rate width; angular noise is multiplied by dt.
    """
    if N < 1 or n_steps < 0 or L <= 0 or dt <= 0:
        raise ValueError("N and L/dt must be positive; n_steps must be nonnegative.")
    if v_0 < 0 or R < 0 or eta < 0:
        raise ValueError("v_0, R, and eta must be nonnegative.")

    rng = np.random.default_rng(seed)
    t = np.arange(n_steps + 1) * dt
    x = np.zeros((len(t), N, 2))
    v = np.zeros_like(x)
    theta = np.zeros((len(t), N))
    x[0] = rng.uniform(0, L, size=(N, 2))
    theta[0] = rng.uniform(0, 2 * np.pi, size=N)
    v[0] = update_velocity(theta[0], v_0)

    for n in range(n_steps):
        theta[n + 1] = update_theta(x[n], theta[n], R, L, eta, rng, dt)
        v[n + 1] = update_velocity(theta[n + 1], v_0)
        x[n + 1] = (x[n] + dt * v[n]) % L

    return x, v, theta, t


In [ ]:
x, v, theta, t = vicsek_model(
    N, L, v_0, R, eta, dt=dt, n_steps=n_steps, seed=seed
)

# x[n, i], v[n, i], theta[n, i]: particle i at time t[n].
x.shape, v.shape, theta.shape, t.shape


In [ ]:
# Visualization: arrows show direction; color encodes theta.
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML, Video, display
from pathlib import Path

alignment = np.hypot(np.mean(np.cos(theta), axis=1),
                      np.mean(np.sin(theta), axis=1))
with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10,
                     "animation.embed_limit": 80}):
    fig = plt.figure(figsize=(9, 6.5), dpi=85, facecolor="#080e1c")
    ax = fig.add_axes([0.08, 0.13, 0.58, 0.67], facecolor="#0c1629")
    chart = fig.add_axes([0.74, 0.36, 0.21, 0.23], facecolor="#0c1629")
    fig.text(0.08, 0.94, "VICSEK / COLLECTIVE MOTION", color="#6ce6ef",
             fontsize=10, weight="bold")
    fig.text(0.08, 0.875, "Simple rules. Moving together.", color="#edf5ff",
             fontsize=23, weight="bold")
    fig.text(0.08, 0.045, "PERIODIC BOX   /   FIXED SPEED   /   LOCAL ALIGNMENT + NOISE",
             color="#879bb5", fontsize=9)
    ax.set(xlim=(0, L), ylim=(0, L), xlabel="x", ylabel="y")
    ax.set_aspect("equal")
    ax.grid(color="#28405a", alpha=0.35, linewidth=0.5)
    for panel in (ax, chart):
        panel.tick_params(colors="#879bb5", labelsize=8)
        panel.xaxis.label.set_color("#b7c9df")
        panel.yaxis.label.set_color("#b7c9df")
        for spine in panel.spines.values():
            spine.set_color("#28405a")
    # Arrow lengths indicate heading, even if v_0 is zero.
    arrow_length = 0.025 * L
    arrows = ax.quiver(x[0, :, 0], x[0, :, 1],
                       arrow_length * np.cos(theta[0]), arrow_length * np.sin(theta[0]),
                       theta[0] % (2 * np.pi), cmap="hsv", clim=(0, 2 * np.pi),
                       angles="xy", scale_units="xy", scale=1, pivot="mid", width=0.005)
    chart.set(xlim=(0, t[-1] if t[-1] > 0 else 1), ylim=(0, 1.03), xlabel="time")
    chart.set_title("COLLECTIVE ALIGNMENT", color="#b7c9df", fontsize=10)
    history, = chart.plot([], [], color="#6ce6ef", lw=1.8)
    fig.text(0.74, 0.22, "1 = aligned\n0 = cancelling directions\nColors show heading",
             color="#879bb5", fontsize=9, linespacing=1.8, va="top")
    readout = fig.text(0.74, 0.78, "", color="#d3e5f8", family="monospace", va="top")
    fps = 24
    frames = np.unique(np.linspace(0, len(t) - 1, min(len(t), 289)).astype(int))

    def update(i):
        arrows.set_offsets(x[i])
        arrows.set_UVC(arrow_length * np.cos(theta[i]), arrow_length * np.sin(theta[i]),
                       theta[i] % (2 * np.pi))
        history.set_data(t[:i + 1], alignment[:i + 1])
        readout.set_text(f"t       {t[i]:6.2f}\nN       {N:6d}\neta     {eta:6.2f}\nalign   {alignment[i]:6.3f}")
        return arrows, history, readout

    animation = FuncAnimation(fig, update, frames=frames, interval=1000 / fps,
                               blit=False, repeat=True)
    player = animation.to_jshtml(default_mode="loop")
    plt.close(fig)

display(HTML(player))


In [ ]:
# Save the live HTML player and a fixed MP4 of the notebook simulation.
import imageio_ffmpeg
from vicsek_player import make_vicsek_player

controlled_player = make_vicsek_player(N, L, v_0, R, eta, dt, n_steps, seed)
Path("vicsek.html").write_text(controlled_player, encoding="utf-8")
with plt.rc_context({"animation.ffmpeg_path": imageio_ffmpeg.get_ffmpeg_exe()}):
    animation.save("vicsek.mp4", writer=FFMpegWriter(fps=fps), dpi=120)

display(Video("vicsek.mp4", embed=True))
